In [ ]:
%pip install shap

In [ ]:
import pandas as pd
import shap
import joblib
import matplotlib.pyplot as plt

rf_model = joblib.load("../artifacts/churn_model.pkl")
feature_names = joblib.load("../artifacts/feature_names.pkl")

df = pd.read_csv("../data/features/customer_segments.csv")
df.head()

In [ ]:
##preparing features
X = df.drop(columns=[
    "CustomerID",
    "Churn"
])

X = pd.get_dummies(
    X,
    drop_first=True
)

X = X.reindex(columns=feature_names, fill_value=0)
X.head()

In [ ]:
##SHAP explainer
explainer = shap.TreeExplainer(rf_model)

shap_values = explainer.shap_values(X)

shap.summary_plot(
    shap_values[:, :, 1],
    X
)

In [ ]:
shap.summary_plot(
    shap_values[:, :, 1],
    X
)

In [ ]:
shap.summary_plot(
    shap_values[:, :, 1],
    X,
    plot_type="bar"
)

In [ ]:
customer_index = 0

explanation = shap.Explanation(
    values=shap_values[customer_index, :, 1],
    base_values=explainer.expected_value[1],
    data=X.iloc[customer_index],
    feature_names=X.columns
)

shap.plots.waterfall(explanation)

In [ ]:
shap.__version__

In [ ]:
# ==========================================================
# SAVE GLOBAL SHAP FEATURE IMPORTANCE
# ==========================================================

import os
import pandas as pd
import numpy as np

# Create artifacts folder if it doesn't already exist
os.makedirs("../artifacts", exist_ok=True)

# ----------------------------------------------------------
# Handle SHAP 0.52 output
# ----------------------------------------------------------

# Your SHAP output is:
# (5630, 42, 2)

if isinstance(shap_values, list):

    # Older SHAP format
    shap_class1 = shap_values[1]

elif len(shap_values.shape) == 3:

    # SHAP 0.52 format
    # samples × features × classes
    shap_class1 = shap_values[:, :, 1]

else:

    shap_class1 = shap_values


# ----------------------------------------------------------
# Calculate mean absolute SHAP importance
# ----------------------------------------------------------

mean_abs_shap = np.abs(
    shap_class1
).mean(axis=0)


# ----------------------------------------------------------
# Create importance dataframe
# ----------------------------------------------------------

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": mean_abs_shap
})


# Sort from most important to least important

shap_importance = (
    shap_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Save as CSV
# ----------------------------------------------------------

shap_importance.to_csv(
    "../artifacts/shap_global_importance.csv",
    index=False
)


print("Global SHAP importance saved successfully!")
print()

print(
    shap_importance.head(15)
)

print()

print(
    "Saved to: ../artifacts/shap_global_importance.csv"
)